In [ ]:
# Locate the repository when opened from any experiment subdirectory.
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents)
             if (p / "smoke_test.py").is_file()
             and (p / "configs/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the PGAE-tau repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Repository: {ROOT}")


In [ ]:
import sys
sys.path.append('.')

In [ ]:
import matplotlib.pyplot as plt
from configs.config import sys_config
from configs.nn_setting import nn_setting
from tqdm import tqdm
from matplotlib import cm

In [ ]:
import torch
import sys

from data.loaders import load_case, load_measurement, load_load_pv, load_dataset
from models.baselines.lstmae import LSTM_AE
from models.baselines.convae import Conv_AE
from data.dataset import rnn_dataset
from torch.utils.data import DataLoader
import numpy as np
from torch import optim
from data.dataset import scaler
from time import time

In [ ]:
class Evaluation:
    def __init__(self, case_class):

        self.nn_setting = nn_setting
        self.case_class = case_class
        self.case_class.gen_torch_constant()                  # Convert the numpy array into torch array for recovery purpose

        # Load the model and trained parameters
        model = Conv_AE()
        model_path = f"saved_model/{sys_config['case_name']}/ConvAE.pt"
        
        model.load_state_dict(torch.load(model_path, map_location=torch.device(nn_setting['device'])))
        
        device = nn_setting['device']
        print(f"Using {device} device")
        model.to(device)
        model.eval()
        print(model)

        self.model = model
        self.device = device

        # Load Validation dataset
        # Valid set
        valid_set = rnn_dataset(mode = 'valid', istransform=True)
        valid_loader = DataLoader(valid_set, batch_size=1, shuffle=False)
        
        # Calculate validation loss
        loss_fn = torch.nn.MSELoss()
        valid_loss = []
        with torch.no_grad():
            for input, output in valid_loader:
                assert np.all(input.numpy() == output.numpy())
                input, output = input.to(device), output.to(device)
                encoded, decoded = model(input)
                loss = loss_fn(decoded, output)
                valid_loss.append(loss.item())

        # detection threshold
        quantile = [0.80,0.82,0.84,0.86,0.88,0.90,0.92,0.94,0.96,0.98,0.999] ### 阈值
        ae_threshold = np.quantile(valid_loss, quantile)
        self.ae_threshold = ae_threshold
        
        self.quantile_idx = 6   # The index of threshold quantile in evaluation
        
        print(f'The detection threshold: {self.ae_threshold}')
        
        # Scaling factor
        scaler_ = scaler()
        self.min = torch.from_numpy(scaler_.min).float().to(nn_setting['device']).requires_grad_(False)
        self.max = torch.from_numpy(scaler_.max).float().to(nn_setting['device']).requires_grad_(False)
    
        self.quantile = quantile
    
    def evaluate(self, test_data):
        if torch.is_tensor(test_data):
            test_data = test_data.float().to(self.device)
        else:
            test_data = torch.tensor(test_data, dtype=torch.float32, device=self.device)    # convert to tensor
            
        assert test_data.shape == (1, self.nn_setting['sample_length'], self.nn_setting['feature_size'])

        #####################################################################
        ## zeros_size 和 lattent_dim的值有关
        ####################################################################

        dim = self.nn_setting['lattent_dim']


        zeros = torch.zeros((1, 1, dim), device=self.nn_setting['device'])
        loss_fn = torch.nn.MSELoss()
        self.model.eval()
        with torch.no_grad():
            encoded, decoded = self.model(test_data)
            loss_recons = loss_fn(decoded, test_data)
            loss_lattent = loss_fn(encoded, zeros)
        
        return encoded.cpu().numpy(), decoded.cpu().numpy(), loss_lattent.item(), loss_recons.item()
    


In [ ]:
"""
Test the performance of attack detection and identification algorithms
"""

from configs.config import sys_config, save_metric
from data.loaders import load_case,load_measurement,load_load_pv

# Load cases, measurement, and load
case_class = load_case()
z_noise_summary, v_est_summary = load_measurement()
load_active, load_reactive, pv_active_, pv_reactive_ = load_load_pv()
test_dataloader_scaled, test_dataloader_unscaled, valid_dataloader_scaled, valid_dataloader_unscaled = load_dataset()


dd_detector = Evaluation(case_class=case_class)  # Instance the data-driven detector
scaler_ = scaler()                               # Instance the scaler class
print(f'Quantile: {dd_detector.quantile[dd_detector.quantile_idx]}')
print(f'Threshold: {dd_detector.ae_threshold[dd_detector.quantile_idx]}')

"""
Metrics
"""
# Attack list
ang_no_list = [1,2,3]
mag_no = 0
ang_str_list = [0.2,0.3]
mag_str = 0

# Attack detection
TP_DDD = {}                   # Record the true positive number of deep learning detector
FP_DDD = []

# Attack recovery
att_strength = {}
recover_deviation = {}        # The difference between recovered state phase angle and the ground truth state phase angle
recovery_ite_no = {}          # The number of iterations in recovery algorithm
recovery_time = []            # The recovery time
residual_bdd = {}         # The bdd residual of the recovered measurement
residual_ddd = {}

"""
FPR of DDD
"""
print('FPR of DDD.')

for idx_, (idx, input, v_est_pre, v_est_last) in tqdm(enumerate(test_dataloader_unscaled)):  
    """
    idx_: starts from 0
    idx: the actual index number in test_dataloader_unscaled
    """
    # Scale
    z_att_noise_scale = scaler_(input)
    encoded, decoded, loss_lattent, loss_recons = dd_detector.evaluate(z_att_noise_scale)
    
    if loss_recons <= dd_detector.ae_threshold[dd_detector.quantile_idx]:
        FP_DDD.append(False)
    else:
        FP_DDD.append(True)
    
"""
TPR of DDD
"""
print('TPR of DDD.')

for ang_no in ang_no_list:
    for ang_str in ang_str_list:
        print(f'({ang_no},{ang_str})')
        """
        Construct the dict keys
        """
        TP_DDD[f'({ang_no},{ang_str})'] = []
        att_strength[f'({ang_no},{ang_str})'] = []
        
        for idx_, (idx, input, v_est_pre, v_est_last) in tqdm(enumerate(test_dataloader_unscaled)):  
            """
            idx_: starts from 0
            idx: the actual index number in test_dataloader_unscaled
            """
            
            v_est_pre = v_est_pre.flatten()
            v_est_last = v_est_last.flatten()  # The ground truth state

            """
            Generate attack
            """
            z_att_noise, v_att_est_last = case_class.gen_fdi_att_dd(z_noise=input, v_est_last=v_est_last, ang_no=ang_no, mag_no=mag_no, ang_str=ang_str, mag_str=mag_str)
            v_att_est_last = torch.from_numpy(v_att_est_last)

            """
            Data-driven detector
            """
            # Scale
            z_att_noise_scale = scaler_(z_att_noise)
            encoded, decoded, loss_lattent, loss_recons = dd_detector.evaluate(z_att_noise_scale)
            
            if loss_recons <= dd_detector.ae_threshold[dd_detector.quantile_idx]:
                TP_DDD[f'({ang_no},{ang_str})'].append(False)
            else:
                TP_DDD[f'({ang_no},{ang_str})'].append(True)

            # Attack strength
            vang_true = np.angle(v_est_last.numpy())
            vang_att = np.angle(v_att_est_last.numpy())
            c_true = (vang_att - vang_true)
            att_strength[f'({ang_no},{ang_str})'].append(np.linalg.norm(c_true,2))
            


save_metric(address = f'metric/{sys_config["case_name"]}/metric_ddd_test.npy', 
            # Data-driven Detector
            TP_DDD = TP_DDD,
            att_strength = att_strength,
            FP_DDD = FP_DDD,
            recover_deviation = recover_deviation,
            residual_bdd = residual_bdd,
            residual_ddd = residual_ddd
            )

In [ ]:
# Load cases, measurement, and load
case_class = load_case()

dd_detector = Evaluation(case_class=case_class)  # Instance the data-driven detector
scaler_ = scaler()     

z_noise_summary, v_est_summary = load_measurement()
load_active, load_reactive, pv_active_, pv_reactive_ = load_load_pv()
test_dataloader_scaled, test_dataloader_unscaled, valid_dataloader_scaled, valid_dataloader_unscaled = load_dataset()

metric_ddd = np.load('metric/case14/metric_ddd_test.npy', allow_pickle=True).item()

## TPR and FPR

In [ ]:
FP_DD_summary = np.zeros(len(dd_detector.quantile))
for input, output in test_dataloader_scaled:
    encoded, decoded, loss_lattent, loss_recons = dd_detector.evaluate(input)
    for idx, quantile in enumerate(dd_detector.quantile):
        if loss_recons >= dd_detector.ae_threshold[idx]:
            FP_DD_summary[idx] += 1

FPR_DD_summary = FP_DD_summary / len(test_dataloader_scaled.dataset)


print(f'DD FPR: {FPR_DD_summary}')

In [ ]:
# Attack list
ang_no_list = [1,2,3]
mag_no = 0
ang_str_list = [0.2,0.3]
mag_str = 0

ROC_dict = {}

for ang_no in ang_no_list:
    # Attack number
    for ang_str in ang_str_list:
        # Attack strength
        
        residual = []
        for _, (idx, input, v_est_pre, v_est_last) in tqdm(enumerate(test_dataloader_unscaled)):
            # Convert data format
            v_est_last = v_est_last.flatten()
            # Attack
            z_att_noise, v_att_est_last = case_class.gen_fdi_att_dd(z_noise=input, v_est_last=v_est_last, ang_no=ang_no, mag_no=mag_no, ang_str=ang_str, mag_str=mag_str) # z_attack_noise: (1,sample_length,feature_size)
            # Scale
            z_att_noise_scale = scaler_(z_att_noise)
            encoded, decoded, _, loss_recons = dd_detector.evaluate(z_att_noise_scale)
            residual.append(loss_recons)
        
        TPR_DDD_summary = []
        # For each of the threshold, find the detection accuracy
        for i, quantile in enumerate(dd_detector.quantile):
            TPR_DDD = np.sum(residual >= dd_detector.ae_threshold[i])/len(residual)
            TPR_DDD_summary.append(TPR_DDD)
            
        ROC_dict[f'({ang_no},{ang_str})'] = TPR_DDD_summary

In [ ]:
plt.figure()
plt.plot(1-np.array(dd_detector.quantile), FPR_DD_summary, label = 'Test Set', color = cm.Dark2(0))
plt.plot(1-np.array(dd_detector.quantile), 1-np.array(dd_detector.quantile), label = 'Valid Set', linestyle = 'dashed', color = cm.Dark2(0))
plt.vlines(x = 1-dd_detector.quantile[dd_detector.quantile_idx], ymin = 0, ymax = 1, colors = 'red', linestyles='--', label='Threshold')
plt.xlabel('FPR')
plt.ylabel('FPR')
plt.fill_between(1-np.array(dd_detector.quantile), 1-np.array(dd_detector.quantile), FPR_DD_summary, color = cm.Dark2(0), alpha = 0.1)
plt.legend()
plt.ylim([0.0,0.31])
plt.yticks(np.arange(0,0.31,0.05))
plt.xlim([0.0,0.2])
plt.grid()
plt.savefig(f'figures/{sys_config["case_name"]}/false_positive.pdf')
plt.show()

In [ ]:
# ROC curve
plt.figure()
for idx, key in enumerate(ROC_dict.keys()):
    plt.plot(FPR_DD_summary, ROC_dict[key], label = key, color = cm.Dark2(idx), linestyle = '-', marker = 'o')
    if idx == 3:
        break
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.grid()
plt.vlines(x = FPR_DD_summary[dd_detector.quantile_idx], ymin = 0, ymax = 1, colors = 'red', linestyles='--', label='Threshold')
plt.legend()
plt.ylim([0.6,1.0])
plt.xlim([0,0.30])
plt.savefig(f'figures/{sys_config["case_name"]}/roc.pdf')
plt.show()

print(f'The validation set FPR: {1-dd_detector.quantile[dd_detector.quantile_idx]}')
print(f'The false positive rate is: {np.sum(metric_ddd["FP_DDD"])/len(metric_ddd["FP_DDD"])}')

In [ ]:
## Compute AUC, F1-score, and accuracy

from sklearn.metrics import auc, precision_score, recall_score, f1_score, accuracy_score
import numpy as np

# Load the metrics saved by the evaluation cells.
try:
    metric_ddd = np.load(f'metric/{sys_config["case_name"]}/metric_ddd_test.npy', allow_pickle=True).item()
except FileNotFoundError:
    print("Warning: metric_ddd_test.npy not found. Please run the evaluation cells (e.g., Cell 4) first to generate metrics.")
    # 如果文件不存在，则跳过计算
    metric_ddd = None


if metric_ddd is not None:
    # --- 1. AUC (Area Under the ROC Curve) ---
    # Integrate the ROC curves computed in the preceding cells.
    for i in range(0,4):
        try:
          model_key = list(ROC_dict.keys())[i]
          tpr_array = ROC_dict[model_key]
          fpr_array = FPR_DD_summary
          roc_auc = auc(fpr_array, tpr_array)
          print(f"AUC-ROC (Attack Case: {model_key}): {roc_auc:.4f}")
        except NameError:
            print("Warning: ROC_dict or FPR_DD_summary not defined. Cannot calculate AUC.")
        except IndexError:
            print("Warning: ROC_dict is empty. Cannot calculate AUC.")



    # --- 2. F1-score 和 ACC (基于选定的阈值) ---
    # 获取选定阈值下的 TP 结果 (所有攻击情况合并)
    all_tpr_results = []
    for key in metric_ddd['TP_DDD'].keys():
        all_tpr_results.extend(metric_ddd['TP_DDD'][key])

    # TP_DDD 记录的是每个攻击样本是否被检测到 (True=TP, False=FN)
    TP_FN_count = len(all_tpr_results) # TP + FN (所有攻击样本总数)
    TP_count = np.sum(all_tpr_results)
    FN_count = TP_FN_count - TP_count

    # FP_DDD 记录的是每个正常样本是否被错误地检测到 (True=FP, False=TN)
    FP_TN_count = len(metric_ddd['FP_DDD']) # FP + TN (所有正常样本总数)
    FP_count = np.sum(metric_ddd['FP_DDD'])
    TN_count = FP_TN_count - FP_count


    # Y_true: 攻击样本=1 (TP+FN), 正常样本=0 (FP+TN)
    Y_true = np.concatenate([np.ones(TP_FN_count), np.zeros(FP_TN_count)])
    
    # Y_pred: 模型检测到攻击=1 (TP+FP), 模型未检测到攻击=0 (FN+TN)
    all_fp_results = metric_ddd['FP_DDD']
    Y_pred = np.concatenate([np.array(all_tpr_results), np.array(all_fp_results)])


    # Compute classification metrics from binary predictions.
    accuracy = accuracy_score(Y_true, Y_pred)
    precision = precision_score(Y_true, Y_pred, zero_division=0)
    recall = recall_score(Y_true, Y_pred, zero_division=0) # Recall = TPR
    f1 = f1_score(Y_true, Y_pred, zero_division=0)

    print("-" * 30)
    print(f"Metrics for selected Threshold (Quantile 0.92):")
    print(f"  Total Attacked Samples (TP+FN): {TP_FN_count}")
    print(f"  Total Normal Samples (FP+TN): {FP_TN_count}")
    print(f"  True Positives (TP): {TP_count}")
    print(f"  False Positives (FP): {FP_count}")
    print(f"  True Negatives (TN): {TN_count}") 
    print(f"  Accuracy (ACC): {accuracy:.4f}") 
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall (TPR): {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print("-" * 30)